In [5]:
import os
import sys
import pandas as pd

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [6]:
# Execution code within notebook context

from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.hypothesis_tests import run_categorical_chi2_test, run_numerical_ttest

# Point directly to your zipped text file path
zip_path = '../data/raw/MachineLearningRating_v3.zip' 

df = load_insurance_data(zip_path)
df = calculate_insurance_metrics(df)

results_summary = []

# Example Test 1: Gender vs Claim Frequency
chi2_g, p_g = run_categorical_chi2_test(df, 'Gender', 'TotalClaims')
results_summary.append({
    'Hypothesis': 'Risk variance across Gender labels',
    'Test Type': 'Chi-Square Contingency',
    'p-value': p_g,
    'Decision': 'Reject H0' if p_g < 0.05 else 'Fail to Reject H0'
})

In [7]:
# Example Test 2: Selected Province Pairs on Claim Severity (Only considering positive claims records)
claims_only = df[df['TotalClaims'] > 0]
if len(df['Province'].unique()) >= 2:
    p1, p2 = df['Province'].value_counts().index[0], df['Province'].value_counts().index[1]
    t_p, p_p = run_numerical_ttest(claims_only, 'Province', p1, p2, 'TotalClaims')
    results_summary.append({
        'Hypothesis': f'Claim Severity Variance ({p1} vs {p2})',
        'Test Type': 'Welch t-test',
        'p-value': p_p,
        'Decision': 'Reject H0' if p_p < 0.05 else 'Fail to Reject H0'
})

print(pd.DataFrame(results_summary).to_markdown(index=False))


| Hypothesis                                        | Test Type              |   p-value | Decision          |
|:--------------------------------------------------|:-----------------------|----------:|:------------------|
| Risk variance across Gender labels                | Chi-Square Contingency |  1        | Fail to Reject H0 |
| Claim Severity Variance (Gauteng vs Western Cape) | Welch t-test           |  0.030599 | Reject H0         |


Task 3: A/B Hypothesis Testing RoadmapImplementation Plan & MethodologyTo evaluate the factors influencing insurance risk, we will analyze two primary Key Performance Indicators (KPIs):Claim Risk / Frequency: The number of claims filed per policyholder.Total Claim Severity: The financial payout amount per claim.Depending on the nature of the data, we apply specific statistical tests:Chi-Square Test of Independence ($\chi^2$): Used for categorical relationships (e.g., comparing claim frequency/counts across categories).T-Test / ANOVA: Used to compare continuous group means (e.g., claim severity across genders or multiple provinces). Note: If the data violates normality assumptions, non-parametric equivalents like Mann-Whitney U or Kruskal-Wallis will be used.The Four Null Hypotheses ($H_0$)Hypothesis 1 (Risk across Provinces):$$H_0: \mu_{\text{Severity, Prov A}} = \mu_{\text{Severity, Prov B}} = \dots = \mu_{\text{Severity, Prov N}}$$There is no significant difference in risk (claim frequency or severity) across different provinces.Hypothesis 2 (Risk between Zip Codes):$$H_0: \mu_{\text{Severity, Zip 1}} = \mu_{\text{Severity, Zip 2}} = \dots = \mu_{\text{Severity, Zip N}}$$There is no significant difference in risk (claim frequency or severity) between different postal codes/zip codes.Hypothesis 3 (Margin/Profitability between Genders):$$H_0: \mu_{\text{Severity, Male}} = \mu_{\text{Severity, Female}}$$There is no significant difference in profit margin or claim risk between male and female policyholders.Hypothesis 4 (Risk across Controls/Vehicle Types):$$H_0: \mu_{\text{Severity, Type 1}} = \mu_{\text{Severity, Type 2}} = \dots = \mu_{\text{Severity, Type N}}$$There is no significant difference in risk across different vehicle types or driver control categories.

Python Code: A/B Hypothesis Testing

In [8]:
!pip install statsmodels

  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 1.2 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.5 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.5 MB 1.5 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/9.5 MB 1.7 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.5 MB 1.7 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/9.5 MB 1.8 MB/s eta 0:00:04
   ------------ --------------------------- 2.9/9.5 MB 1.8 MB/s eta 0:00:04
   -------------- ------------------------- 3.4/9.5 MB 1.8 MB/s eta 0:00:04
   ---------------- ----------------------- 3.9/9.

In [9]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# --- Simulated Data Setup (Replace with your actual dataset) ---
np.random.seed(42)
n_samples = 1000
df = pd.DataFrame({
    'Province': np.random.choice(['Province_A', 'Province_B', 'Province_C'], n_samples),
    'ZipCode': np.random.choice(['ZIP_01', 'ZIP_02', 'ZIP_03', 'ZIP_04'], n_samples),
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'VehicleType': np.random.choice(['Sedan', 'SUV', 'Truck'], n_samples),
    'Claim_Frequency': np.random.poisson(lam=0.2, size=n_samples), # Categorical count risk
    'Claim_Severity': np.random.exponential(scale=500, size=n_samples) # Continuous financial risk
})
# Ensure severity is 0 if no claim was made
df.loc[df['Claim_Frequency'] == 0, 'Claim_Severity'] = 0


# --- Hypothesis 1: Risk across Provinces (ANOVA for Severity) ---
print("--- Hypothesis 1: Risk across Provinces ---")
model_prov = ols('Claim_Severity ~ C(Province)', data=df[df['Claim_Severity'] > 0]).fit()
anova_prov = sm.stats.anova_lm(model_prov, typ=2)
print(f"ANOVA p-value for Severity across Provinces: {anova_prov['PR(>F)'].iloc[0]:.4f}\n")


# --- Hypothesis 2: Risk between Zip Codes (ANOVA for Severity) ---
print("--- Hypothesis 2: Risk between Zip Codes ---")
model_zip = ols('Claim_Severity ~ C(ZipCode)', data=df[df['Claim_Severity'] > 0]).fit()
anova_zip = sm.stats.anova_lm(model_zip, typ=2)
print(f"ANOVA p-value for Severity across Zip Codes: {anova_zip['PR(>F)'].iloc[0]:.4f}\n")


# --- Hypothesis 3: Risk between Genders (Independent t-test for Severity) ---
print("--- Hypothesis 3: Risk between Genders ---")
male_severity = df[(df['Gender'] == 'Male') & (df['Claim_Severity'] > 0)]['Claim_Severity']
female_severity = df[(df['Gender'] == 'Female') & (df['Claim_Severity'] > 0)]['Claim_Severity']

t_stat, t_pval = stats.ttest_ind(male_severity, female_severity, equal_var=False)
print(f"T-test p-value for Severity between Genders: {t_pval:.4f}\n")


# --- Hypothesis 4: Risk across Vehicle Types (Chi-Square for Claim Frequency) ---
print("--- Hypothesis 4: Risk across Vehicle Types ---")
# Create a contingency table of Vehicle Type vs Claim Occurred (Yes/No)
df['Claim_Occurred'] = df['Claim_Frequency'].apply(lambda x: 1 if x > 0 else 0)
contingency_table = pd.crosstab(df['VehicleType'], df['Claim_Occurred'])

chi2, chi2_pval, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-Square p-value for Claim Frequency across Vehicle Types: {chi2_pval:.4f}\n")

--- Hypothesis 1: Risk across Provinces ---
ANOVA p-value for Severity across Provinces: 0.9946

--- Hypothesis 2: Risk between Zip Codes ---
ANOVA p-value for Severity across Zip Codes: 0.9299

--- Hypothesis 3: Risk between Genders ---
T-test p-value for Severity between Genders: 0.4622

--- Hypothesis 4: Risk across Vehicle Types ---
Chi-Square p-value for Claim Frequency across Vehicle Types: 0.6446

